# Avance Fase 3 — Semana 2
## Núcleo algorítmico, eficiencia y programación orientada a objetos

**Curso:** MCDI500 Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e Inteligencia Artificial, UNAB
**Proyecto:** Morosidad en patentes por no uso de derechos de aprovechamiento de aguas (Grupo 9)
**Docente:** Dr. Omar Salinas Silva

En la Fase 2 el equipo construyó un pipeline que funciona, pero como una secuencia de celdas. En esta fase ese
mismo procesamiento se **reorganiza en módulos y clases** (carpeta `src/`), sin cambiar su resultado, y se
demuestra con evidencia que el resultado es el mismo.

| Sección | Qué se hace | Criterio |
|---|---|---|
| 1. Configuración | Parámetros del proyecto y carga de los módulos de `src/` | V |
| 2. Carga y limpieza funcional | Funciones que reciben un DataFrame y devuelven uno nuevo | II.a |
| 3. Pipeline de clases | Clase base `Transformador`, 4 transformadores y `Pipeline` | II.b, III.a |
| 4. Comparación con F2 | El pipeline de clases reproduce el archivo procesado de F2 | II.c |
| 5. Patrón Strategy | Cambiar el tratamiento del año faltante sin tocar el pipeline | III.b |
| 6. Partición y salidas para F4 | Partición estratificada, semilla fija y archivos en `data/processed` | Sección 5 de la guía |
| 7. Validación, eficiencia y diseño | Pruebas, mediciones, función recursiva y arquitectura (módulos de las otras ramas) | II.c, II.d, II.e |

## 1. Configuración

Todos los parámetros del proyecto están en la celda siguiente. El notebook localiza la raíz del proyecto
por sí mismo, así que no usa rutas propias de ningún computador, y agrega `src/` al camino de búsqueda de Python
para **importar** los módulos en lugar de redefinir las clases en celdas.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from sklearn.model_selection import train_test_split

# --- Parámetros del proyecto -------------------------------------------------
SEMILLA = 42                      # semilla fija: la partición es reproducible
PROPORCION_PRUEBA = 0.2           # 80 % entrenamiento, 20 % prueba
COLUMNA_OBJETIVO = "moroso"       # 1 = tiene saldo impago (SALDO > 0)
RUTA_CRUDO = "data/raw/tb_patentes_aguas_december_2025_data.csv"
RUTA_F2 = "data/processed/tb_patentes_aguas_december_2025_data.csv"   # salida de F2 (no se modifica)
DIR_PROCESADO = "data/processed"  # aquí se guardan las salidas de F3 para F4
# Columnas que no deben entrar a un modelo: la variable objetivo se define con SALDO (fuga de información)
COLUMNAS_EXCLUIDAS_MODELADO = ["SALDO", "PAGADO", "LOG_SALDO", "LOG_PAGADO"]


def localizar_raiz(inicio=None):
    """Devuelve la raíz del proyecto: si el kernel está en una carpeta de fase, la superior."""
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ / "src"))   # permite: from carga import ...

from carga import (cargar_datos, preparar_crudo, describir_columnas, guardar_entregables)
from transformadores import (Transformador, TratamientoAnio, CodificadorOneHot, TransformacionLog,
                             CreacionMoroso, RellenarConValor, DejarNulo, MedianaPorTesoreria)
from pipeline import Pipeline, crear_pipeline_f2

print("Raíz del proyecto:", RAIZ)
print("Módulos importados desde:", RAIZ / "src")

## 2. Carga y limpieza funcional (II.a)

`src/carga.py` reúne funciones con una sola tarea, con *docstring*, parámetros explícitos y errores controlados.
Todas **reciben un DataFrame y devuelven uno nuevo**: el original no se modifica. Se comprueba más abajo.

- `cargar_datos(ruta)`: lee el CSV; si no existe, lanza un `FileNotFoundError` que indica dónde debe estar.
- `eliminar_columnas(df)`: quita los identificadores, fechas y variables de alta cardinalidad que F2 descartó.
- `eliminar_filas_nulas(df, columna)`: quita las filas sin tesorería regional.
- `preparar_crudo(df)`: combina las dos anteriores, tal como en F2.

In [ ]:
crudo = cargar_datos(RAIZ / RUTA_CRUDO)
copia_de_control = crudo.copy()

base = preparar_crudo(crudo)

# Verificación: el original no se modificó y la limpieza descartó solo lo esperado
assert_frame_equal(crudo, copia_de_control)
print(f"Archivo original : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print(f"Tras la limpieza : {base.shape[0]:,} filas x {base.shape[1]} columnas")
print(f"Filas descartadas: {crudo.shape[0] - base.shape[0]} (sin TESORERIA_REGIONAL)")
print("El DataFrame original no se modificó: OK")
base.head(3)

## 3. Pipeline de clases (II.b y III.a)

Cada paso del pipeline de F2 pasó a ser una **clase** que hereda de `Transformador` (`src/transformadores.py`).
La clase base define `ajustar` (aprender del entrenamiento) y `transformar` (aplicar lo aprendido), y guarda
su estado en un atributo **privado**. El `Pipeline` (`src/pipeline.py`) recibe la lista de pasos y llama a
todos con el mismo método: eso es **polimorfismo**; agregar un paso no obliga a tocar el pipeline.

| Clase | Qué hace | Por qué es pertinente |
|---|---|---|
| `TratamientoAnio` | Marca con `anio_faltante` los años que faltan y los trata según una estrategia | Hay 691 patentes sin año; perder esa información sesgaría la comparación por año |
| `CodificadorOneHot` | Convierte `TESORERIA_REGIONAL` y `TIPO_PERSONA` en columnas binarias | Son nominales sin orden natural; un número entero inventaría un orden |
| `TransformacionLog` | Crea `LOG_*` con `log1p` de las cuatro variables en pesos | Los montos son muy asimétricos; el logaritmo hace comparables los gráficos |
| `CreacionMoroso` | Crea la variable objetivo `moroso = (SALDO > 0)` | Es la pregunta del proyecto: quién tiene saldo impago |

In [ ]:
# Herencia y polimorfismo, comprobados
pasos = [TratamientoAnio(), CodificadorOneHot("TESORERIA_REGIONAL"), TransformacionLog(), CreacionMoroso()]
for paso in pasos:
    print(f"{type(paso).__name__:<20} hereda de Transformador: {isinstance(paso, Transformador)}")

# El estado aprendido es privado (atributo con doble guion bajo): no se accede por su nombre simple.
# Desde afuera se consulta con la propiedad .parametros, que devuelve una copia.
try:
    pasos[0].__estado
except AttributeError:
    print("\nEl estado interno es privado: se consulta con .parametros, que devuelve una copia")
print("Parámetros antes de ajustar:", pasos[0].parametros, "| ajustado:", pasos[0].ajustado)

In [ ]:
# El pipeline de F2 reorganizado en clases: los pasos y su orden
pipeline_f2 = crear_pipeline_f2()
for i, paso in enumerate(pipeline_f2.pasos, start=1):
    print(f"{i}. {type(paso).__name__}")

### Efecto medido de cada paso

Cada decisión se justifica con una medición hecha sobre el propio conjunto, no con una afirmación.

In [ ]:
from scipy.stats import skew

procesado_completo = crear_pipeline_f2().ajustar_transformar(base)

efectos = pd.DataFrame([
    {"paso": "TratamientoAnio",
     "medición": "filas con año faltante -> filas marcadas con anio_faltante",
     "antes": int(base["ANIO_PATENTE"].isna().sum()),
     "después": int(procesado_completo["anio_faltante"].sum())},
    {"paso": "CodificadorOneHot",
     "medición": "columnas nominales -> columnas binarias",
     "antes": 2,
     "después": int(procesado_completo.filter(regex="^(teso_|persona_)").shape[1])},
    {"paso": "TransformacionLog",
     "medición": "asimetría de MONTO (sesgo)",
     "antes": round(float(skew(base["MONTO"])), 2),
     "después": round(float(skew(procesado_completo["LOG_MONTO"])), 2)},
    {"paso": "TransformacionLog",
     "medición": "asimetría de SALDO (sesgo)",
     "antes": round(float(skew(base["SALDO"])), 2),
     "después": round(float(skew(procesado_completo["LOG_SALDO"])), 2)},
    {"paso": "CreacionMoroso",
     "medición": "proporción de morosos (clase positiva)",
     "antes": "no existía",
     "después": f"{procesado_completo['moroso'].mean():.1%}"},
])
efectos

## 4. Comparación con F2 (II.c)

La prueba que más rinde: ejecutar el pipeline de clases sobre el archivo original y comparar su salida con el
archivo procesado en F2 usando `pandas.testing.assert_frame_equal`, que verifica filas, columnas, orden, tipos
y valores. Si coinciden, hay evidencia objetiva de que reorganizar el código **no cambió nada**.

In [ ]:
f2 = pd.read_csv(RAIZ / RUTA_F2)
salida_clases = crear_pipeline_f2().ajustar_transformar(base)

print(f"Pipeline de clases: {salida_clases.shape[0]:,} filas x {salida_clases.shape[1]} columnas")
print(f"Procesado de F2   : {f2.shape[0]:,} filas x {f2.shape[1]} columnas")

assert salida_clases.shape[0] == 46433, "El caso normal debe conservar las 46.433 filas"
assert_frame_equal(salida_clases, f2)     # falla con un error descriptivo si algo difiere
print("\nassert_frame_equal(pipeline de clases, procesado de F2): IGUALES")

## 5. Patrón de diseño: Strategy (III.b)

**Patrón elegido: Strategy.** El tratamiento del año faltante admite varias formas válidas (dejarlo nulo,
rellenar con un valor, imputar la mediana por tesorería). Cada una es una clase con la misma interfaz
(`EstrategiaAnio`), y `TratamientoAnio` recibe la que se quiera. **El pipeline no cambia** al elegir otra:
solo cambia el argumento. Con `RellenarConValor(0)` se reproduce exactamente F2.

In [ ]:
estrategias = {
    "RellenarConValor(0)  [F2]": RellenarConValor(0),
    "DejarNulo": DejarNulo(),
    "MedianaPorTesoreria": MedianaPorTesoreria(),
}

def es_igual(a, b):
    # True si assert_frame_equal no encuentra diferencias (misma comparación que la sección 4)
    try:
        assert_frame_equal(a, b)
        return True
    except AssertionError:
        return False


resumen = []
for nombre, estrategia in estrategias.items():
    resultado = crear_pipeline_f2(estrategia_anio=estrategia).ajustar_transformar(base)
    resumen.append({"estrategia": nombre,
                    "nulos en ANIO_PATENTE": int(resultado["ANIO_PATENTE"].isna().sum()),
                    "columnas": resultado.shape[1],
                    "idéntico a F2": es_igual(resultado, f2)})
pd.DataFrame(resumen)

## 6. Partición y salidas para F4

La clase positiva (`moroso = 1`) es la mayoritaria, cerca del 77 %. Por eso la partición es **estratificada por
`moroso`**, y con **semilla fija** para que cualquiera obtenga la misma. Como `moroso` la crea el propio pipeline,
la estratificación usa `SALDO > 0` calculado sobre el conjunto limpio (es la misma definición).

Lo que se aprende de los datos (las categorías de tesorería, las medianas) se **ajusta solo con el
entrenamiento** y se aplica a la prueba, para que la prueba no influya en nada.

In [ ]:
estrato = (base["SALDO"] > 0).astype(int)
entrenamiento, prueba = train_test_split(
    base, test_size=PROPORCION_PRUEBA, stratify=estrato, random_state=SEMILLA)

pipeline_final = crear_pipeline_f2()
entrenamiento_final = pipeline_final.ajustar_transformar(entrenamiento)   # ajusta SOLO con entrenamiento
prueba_final = pipeline_final.transformar(prueba)                         # aplica lo aprendido

print(pd.DataFrame({
    "conjunto": ["entrenamiento", "prueba", "total"],
    "filas": [len(entrenamiento_final), len(prueba_final), len(entrenamiento_final) + len(prueba_final)],
    "% morosos": [f"{entrenamiento_final['moroso'].mean():.2%}", f"{prueba_final['moroso'].mean():.2%}",
                  f"{procesado_completo['moroso'].mean():.2%}"],
}).to_string(index=False))

In [ ]:
diccionario = describir_columnas(
    entrenamiento_final, originales=base.columns, objetivo=COLUMNA_OBJETIVO,
    excluidas=COLUMNAS_EXCLUIDAS_MODELADO)

entregables = guardar_entregables(entrenamiento_final, prueba_final, pipeline_final,
                                  diccionario, RAIZ / DIR_PROCESADO)

print("ARTEFACTOS PARA LA FASE 4")
for nombre, ruta in entregables.items():
    print(f"  {nombre:<14} {ruta.relative_to(RAIZ)}  ({ruta.stat().st_size / 1024:,.0f} KB)")
diccionario.head(12)

### Verificación de entrega

Cinco comprobaciones antes de pasar los archivos a la Fase 4. Si alguna falla, el problema es de esta fase.

In [ ]:
releido_ent = pd.read_csv(entregables["entrenamiento"])
releido_pru = pd.read_csv(entregables["prueba"])

controles = [
    ("Se conservan las 46.433 filas", len(releido_ent) + len(releido_pru) == 46433),
    ("Entrenamiento y prueba con el mismo esquema", list(releido_ent.columns) == list(releido_pru.columns)),
    ("Sin valores nulos", int(releido_ent.isna().sum().sum() + releido_pru.isna().sum().sum()) == 0),
    ("La variable objetivo está completa y es 0/1", set(releido_ent[COLUMNA_OBJETIVO].unique()) <= {0, 1}),
    ("Proporción de morosos similar en ambos conjuntos",
     abs(releido_ent[COLUMNA_OBJETIVO].mean() - releido_pru[COLUMNA_OBJETIVO].mean()) < 0.005),
    ("Lo guardado se relee idéntico a lo guardado",
     releido_ent.shape == entrenamiento_final.shape and releido_pru.shape == prueba_final.shape),
]
tabla = pd.DataFrame([{"control": c, "estado": "OK" if ok else "FALLA"} for c, ok in controles])
assert all(ok for _, ok in controles), "Alguna verificación de entrega falló"
tabla

## 7. Validación, eficiencia y estructura (II.c, II.d, II.e y III.b)

Esta sección usa los módulos que aportaron las otras ramas del equipo, importándolos desde `src/` y `tests/` sin redefinirlos aquí.

### 7.1 Validación técnica (II.c): pruebas automatizadas

La carpeta `tests/` (rama `f3-pruebas`) reúne casos **normales**, **límite** (una fila, saldo exactamente 0, categoría nueva) y **excepciones** (`RuntimeError`, `KeyError`, `TypeError`, `ValueError`, `FileNotFoundError`). Se ejecutan con `pytest`; la última prueba compara el pipeline con el archivo de F2.

In [ ]:
import subprocess

resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
                           cwd=RAIZ, capture_output=True, text=True)
print(resultado.stdout.strip())
if resultado.returncode != 0:
    print(resultado.stderr.strip())
assert resultado.returncode == 0, "Alguna prueba falló (si dice 'No module named pytest': pip install pytest)"

### 7.2 Eficiencia (II.d): tiempo, memoria y complejidad

`src/medicion.py` (rama `f3-mediciones`) compara dos tareas sobre las 46.433 filas: `moroso` con un bucle frente a la versión vectorizada, y la tasa de morosidad por tesorería con un diccionario frente a `groupby`. El tiempo se mide con `timeit` (se conserva el **menor** de varias repeticiones) y la memoria con `tracemalloc`. El programa se detiene si las dos versiones no dan el mismo resultado.

In [ ]:
import medicion

tabla_mediciones = medicion.main()

**Lectura de los resultados.** Las cuatro versiones son O(n): la diferencia está en el factor constante, porque el recorrido en Python es mucho más lento que en C. Para `moroso` se usa la versión vectorizada (más rápida y con menos memoria). Para la tasa por tesorería se usa `groupby`: es más rápida aunque usa más memoria que el diccionario, y con 46 mil filas esa diferencia no es una restricción. El detalle está en `F3/mediciones.md`.

### 7.3 Diseño estructurado (II.e): función recursiva sobre los metadatos

`src/recursion.py` (rama `f3-documentacion`) convierte los metadatos del proyecto en un diccionario anidado y lo recorre recursivamente. **Caso base:** el valor no es un diccionario (una hoja), que se revisa directamente. **Caso recursivo:** el valor es un diccionario, y la función se llama de nuevo sobre cada hijo acumulando la ruta.

In [ ]:
import recursion

metadatos = recursion.cargar_metadatos(RAIZ / "docs" / "metadatos_proyecto.csv")
print("Campos en el árbol de metadatos:", recursion.contar_campos(metadatos))
print("Campos vacíos o pendientes:", recursion.buscar_pendientes(metadatos) or "ninguno")

# La misma función sobre un ejemplo con huecos, para ver que detecta los pendientes
ejemplo = {"datos": {"archivo": "x.csv", "fuente": {"autor": "", "url": "PENDIENTE"}}}
print("Ejemplo con huecos:", recursion.buscar_pendientes(ejemplo))

### 7.4 Arquitectura (III.b)

La tabla de componentes, el patrón Strategy justificado, el diagrama de flujo y la escalabilidad al archivo de 2026 están en `F3/arquitectura.md`; el contenido de la carpeta y cómo ejecutar este notebook, en `F3/README.md`.

## Conclusiones y trazabilidad

- El pipeline de F2 quedó reorganizado en una clase base y cuatro transformadores, y su salida es **idéntica** a la de F2 (46.433 filas x 31 columnas).
- Lo aprendido de los datos se ajusta solo con el entrenamiento; la prueba solo recibe lo aprendido.
- Las pruebas automatizadas (`tests/`) pasan, e incluyen la comparación con F2.
- La eficiencia se midió con `timeit` y `tracemalloc`; se usan la versión vectorizada y `groupby`.
- Las salidas para F4 quedan en `data/processed/`: `entrenamiento_f3.csv`, `prueba_f3.csv`, `diccionario_f3.csv` y `parametros_f3.csv`.
- Código en `src/` (`carga.py`, `transformadores.py`, `pipeline.py`, `medicion.py`, `recursion.py`); este notebook solo los importa.

## Declaración de uso de inteligencia artificial

Conforme a la integridad académica de la UNAB, el equipo declara que utilizó IA generativa (Anthropic, 2026) como herramienta de apoyo en esta fase para: (a) orientar el trabajo con Git y GitHub (ramas, commits y pull requests); (b) redactar borradores del código de apoyo (`src/medicion.py`, `src/recursion.py` y las pruebas de `tests/`) y de la documentación (`F3/README.md`, `F3/arquitectura.md`, `F3/mediciones.md`); (c) completar esta sección 7; y (d) redactar y revisar el informe.

La IA no reemplazó el juicio del equipo: cada integrante ejecutó el código en su equipo, las mediciones provienen de esas ejecuciones, los cambios se revisaron y aprobaron mediante pull requests, y las decisiones de diseño y la verificación de resultados son responsabilidad del grupo.

Referencia: Anthropic. (2026). *Claude* [Modelo de lenguaje de gran tamaño]. https://www.anthropic.com